In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

# Suppress torchvision download progress bars
torchvision.set_image_backend('accimage') # Optional backend optimization
datasets.MNIST.mirrors = [
    'https://ossci-datasets.s3.amazonaws.com/mnist/'
]

# Disable tqdm / progress bars globally for torchvision downloads
import tqdm
from unittest.mock import patch
patch('tqdm.tqdm', lambda *args, **kwargs: args[0] if args else None).start()

<function __main__.<lambda>(*args, **kwargs)>

In [3]:
# 1. Hyperparameters
BATCH_SIZE = 64
LEARNING_RATE = 0.001
EPOCHS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
# 2. Data Loading and Preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


In [5]:
# 3. Model Architecture Definition
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.network(x)

model = MLP().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [6]:
# 4. Model Training
print("Starting Training...\n")
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
    
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch}/{EPOCHS}] - Loss: {epoch_loss:.4f}")


Starting Training...

Epoch [1/5] - Loss: 0.2780
Epoch [2/5] - Loss: 0.1142
Epoch [3/5] - Loss: 0.0800
Epoch [4/5] - Loss: 0.0617
Epoch [5/5] - Loss: 0.0524


In [7]:
# 5. Evaluation Metrics
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro')

print("\n--- Model Evaluation Results ---")
print(f"Overall Accuracy : {accuracy * 100:.2f}%")
print(f"Macro Precision  : {precision:.4f}")
print(f"Macro Recall     : {recall:.4f}")
print(f"Macro F1-Score   : {f1:.4f}\n")
print("Detailed Classification Report:\n")
print(classification_report(all_labels, all_preds, digits=4))


--- Model Evaluation Results ---
Overall Accuracy : 97.19%
Macro Precision  : 0.9722
Macro Recall     : 0.9717
Macro F1-Score   : 0.9717

Detailed Classification Report:

              precision    recall  f1-score   support

           0     0.9838    0.9898    0.9868       980
           1     0.9886    0.9894    0.9890      1135
           2     0.9871    0.9651    0.9760      1032
           3     0.9450    0.9871    0.9656      1010
           4     0.9842    0.9491    0.9663       982
           5     0.9794    0.9608    0.9700       892
           6     0.9790    0.9749    0.9770       958
           7     0.9857    0.9387    0.9616      1028
           8     0.9504    0.9836    0.9667       974
           9     0.9391    0.9782    0.9583      1009

    accuracy                         0.9719     10000
   macro avg     0.9722    0.9717    0.9717     10000
weighted avg     0.9724    0.9719    0.9719     10000

